# 96 — Grand Ensemble v8

ElasticNetCV meta-learner over ALL OOF files, using the proper nested-CV approach from nb86.

Includes all new models from nb86–nb95:
- nb87: ChEMBL NR biological fingerprint
- nb88: 3D conformer shape
- nb89: PXR pharmacophore
- nb90: Tox21 bio-FP
- nb91: cliff-adaptive blend
- nb92: multi-NR transfer
- nb93: Kaggle GPU large Chemprop (if available)
- nb94: MolFormer fine-tune (if available)
- nb95: all-feature fusion

Uses nested-CV stacking (no in-sample leakage).

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from sklearn.linear_model import ElasticNetCV

EXCLUDE = {"aux_features","grand_v6","grand_v6b","grand_v6c","grand_v7",
           "grand15","grand18","grand23","grand24","grand25",
           "creative_mega_ensemble","nested_cv_ensemble",
           "cliff_role_proba","chemprop_cliff_mem_proba",
           "chemprop_chembl_nr_multitask","per_fp_stack",
           "cliff_adaptive_blend"}  # test-time only

oof_files = sorted(DATA_PROCESSED.glob("oof_*.npy"))
oofs, tes, names = [], [], []
for fp in oof_files:
    name = fp.stem.replace("oof_","")
    if name in EXCLUDE: continue
    te_fp = DATA_PROCESSED / f"te_oof_{name}.npy"
    try:
        arr = np.load(fp)
        if arr.ndim > 1: arr = arr[:,0]
        if len(arr) != len(y_tr): continue
        te_v = np.load(te_fp) if te_fp.exists() else None
        if te_v is None or len(te_v) != 513: continue
        if te_v.ndim > 1: te_v = te_v[:,0]
        if te_v.std() < 0.4 * y_tr.std(): continue
        arr[~np.isfinite(arr)] = y_tr.mean()
        te_v[~np.isfinite(te_v)] = float(np.nanmean(te_v))
        oofs.append(arr); tes.append(te_v); names.append(name)
    except Exception as e:
        print(f"  skip {name}: {e}")

OOF_stack = np.column_stack(oofs)
TE_stack  = np.column_stack(tes)
print(f"v8 ensemble: {len(names)} models, stack {OOF_stack.shape}")


v8 ensemble: 30 models, stack (4139, 30)


In [5]:
# Proper nested-CV stacking
print("=== Nested-CV Grand Ensemble v8 ===", flush=True)
oof_v8 = np.full(len(y_tr), np.nan)

for k, (tr_idx, va_idx) in enumerate(splits):
    meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]
    meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    meta.fit(OOF_stack[meta_tr_idx], y_tr[meta_tr_idx])
    oof_v8[va_idx] = meta.predict(OOF_stack[va_idx])
    print(f"  fold {k+1}  val_RAE={rae(y_tr[va_idx], oof_v8[va_idx]):.4f}", flush=True)

m_v8 = full_metrics(y_tr, oof_v8, cliff_pairs, "grand_v8")
m_v8_a = full_metrics(y_tr[active_mask], oof_v8[active_mask], label="v8 [active]")
print(f"\nGrand v8 OOF RAE: {m_v8['RAE']:.4f}")
print(pd.DataFrame([m_v8, m_v8_a], index=["overall","active"]).round(4).to_string())


=== Nested-CV Grand Ensemble v8 ===


  fold 1  val_RAE=0.2521


  fold 2  val_RAE=0.2756


  fold 3  val_RAE=0.3122


  fold 4  val_RAE=0.2900


  fold 5  val_RAE=0.3038


  [grand_v8] RAE=0.2843 MAE=0.2586 R²=0.8402 r=0.9166 ρ=0.8889 τ=0.7443
  [v8 [active]] RAE=2.0164 MAE=0.4228 R²=-3.8352 r=0.2450 ρ=0.1980 τ=0.1521

Grand v8 OOF RAE: 0.2843
            RAE     MAE      R2  Pearson  Spearman  Kendall
overall  0.2843  0.2586  0.8402   0.9166    0.8889   0.7443
active   2.0164  0.4228 -3.8352   0.2450    0.1980   0.1521


In [6]:
meta_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
meta_final.fit(OOF_stack, y_tr)
te_preds = np.clip(meta_final.predict(TE_stack), y_tr.min()-0.5, y_tr.max()+0.5)

coef_df = pd.DataFrame({"model": names, "weight": meta_final.coef_}).sort_values("weight", ascending=False)
print("Non-zero weights:")
print(coef_df[coef_df.weight.abs() > 1e-6].to_string(index=False))

np.save(DATA_PROCESSED/"oof_grand_v8.npy", oof_v8)
np.save(DATA_PROCESSED/"te_oof_grand_v8.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"96_grand_ensemble_v8.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** Grand v8 OOF RAE: {m_v8['RAE']:.4f} ***")


Non-zero weights:
                  model    weight
 delta_similarity_tiers  0.838630
               grand_v8  0.164323
      multi_fp_ensemble  0.127622
               delta_ml  0.067545
 lgbm_chembl_pxr_direct  0.061054
     3d_shape_conformer  0.057037
     bio_nr_fingerprint  0.038274
     all_feature_fusion  0.038248
lgbm_crc_singleconc_fdr  0.029573
           tox21_bio_fp  0.011161
     scaffold_aware_knn  0.003643
     delta_chemprop_cpu -0.392584
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\96_grand_ensemble_v8.csv
Test: min=3.09 med=5.00 max=6.26

*** Grand v8 OOF RAE: 0.2843 ***
